# Rapport Statistique desciptive

---

MEDJDKOUH Khaled, ing3 macs

RAZAFINDRAKOTO Davidson Lova, ing3 macs

---

## But du papier :


* ## FNN ([Feedfoward Neural Network](https://en.wikipedia.org/wiki/Feedforward_neural_network))

<img src="photo/FNN.png" />

* ## BNN (Bayesian Neural Network) 

On dispose de données $D = (x,y) \in \mathcal{X} \times \mathcal{Y} \in \mathbb{R}^m \times \mathbb{R}^l$.

(Ici on note que $(x,y)$ est condiré comme la variable aléatoire qui resume tout les couples $(x(\omega), y(\omega))$)

On se donne une modèle $\mathcal{M}$ (ici un FNN) de paramètre $\theta = \{w ,b\}$ (où $w$ sont les poids et $b$ le biais).

Le BNN consiste à estimer la loi a posteriori de $\theta$ sachant celles du modèle $\mathcal{M}$ et des données $D$. 

On note la densité de cette loi $p(\theta |D , \mathcal{M})$

La manière standard d'estimer cette loi est de la déduire de la formule de Bayes $p(\theta |D , \mathcal{M}) = \frac{p(D| \theta, \mathcal{M}) p(\theta|\mathcal{M})}{p(D|\mathcal{M})}$

* $p(\theta| D, \mathcal{M})$ : distribution a posteriori de $\theta$ <span style="color:red">à estimer</span>
* $p(\theta|\mathcal{M})$ : distribution a priori de $\theta$ <span style="color:red"> donner au début</span>
* $p(D| \theta, \mathcal{M})$ : la vraissemblance des données sachant les paramètres $\theta$ et le modèle $ \mathcal{M}$  <span style="color:red"> couteux à estimer</span>
* $p(D|\mathcal{M})$ : Distribution des données sachant le modèle <span style="color:orange"> constante muliplicative</span>

    Des méthodes existent pour estimer cette loi sans l'évaluation de  $p(D| \theta, \mathcal{M})$ qui serviront de benchmark à la méthode proposé ici.

* ### ABC ([Approximate Bayesian Computation](https://en.wikipedia.org/wiki/Approximate_Bayesian_computation))

    La méthode ABC consiste à estimer $p(\theta |D , \mathcal{M})$ sans passer par l'évaluation de la fonction de vraissemble qui peut s'averer couteuse.   

    On se donne $ \hat{y} = f(x, \theta)$ qui provient de l'évaluation de $x$ par le modèle M (feedforward) avec $\theta \sim p(\theta, \mathcal{M})$

    La formule de Bayes nous donne $p( \theta, \hat{y} | D, \mathcal{M}) \propto p( D | \hat{y}, \theta, \mathcal{M})p( \hat{y}| \theta,\mathcal{M})p( \theta| \mathcal{M}) $

    On simule la loi à droite par une méthode de rejet en simulant $\theta \sim p(\theta |M)$, $\hat{y} \sim p( \hat{y}| \theta,\mathcal{M})$ ensuite on accepte $D = (x,y)$ seulement si $y = \hat{y}$.

    Cela peut s'averer couteux(voir impossible en temps de calcul raisonable) et on introduit alors un seuil $\epsilon$ et on remplace l'égalité par $|y - \hat{y}| < \epsilon$

    On obtient alors $p_{\epsilon} (\theta, \hat{y}| D, \mathcal{M}) \propto \mathbb{I}_{\mathcal{N}_\epsilon (D)} (\hat{y}) p( \hat{y}| \theta,\mathcal{M})p( \theta| \mathcal{M}) $ où $\mathcal{N}_\epsilon (D= (x,y)) = \left\{ \hat{y} \in \mathcal{Y}, \rho(\eta(y), \eta(\hat{y})) \leq \epsilon\right\}  $

    Avec $\eta$ une statistique qui résume une loi et $\rho$ une mesure de similarité.

    Pour revenir à la loi qu'on voulais estimer il suffit maintenant d'intégré par rapport à $\hat{y}$ 

    $p_{\epsilon}( \theta | D, \mathcal{M}) \propto \int_{\mathcal{Y}} p_{\epsilon}( \theta , \hat{y}| D, \mathcal{M}) d \hat{y} \propto \int_{\mathcal{Y}} \mathbb{I}_{\mathcal{N}_\epsilon (D)} (\hat{y}) p( \hat{y}| \theta,\mathcal{M}) p( \theta| \mathcal{M}) d \hat{y} = p( \theta| \mathcal{M}) \int_{\mathcal{Y}} \mathbb{I}_{\mathcal{N}_\epsilon (D)} (\hat{y})  p( \hat{y}| \theta,\mathcal{M}) d \hat{y} = \mathbb{P} (\hat{y} \in \mathcal{N}_{\epsilon} (D)| \theta, \mathcal{M}) p( \theta| \mathcal{M})$

    Pour $\epsilon$ suffisament petit, $p_{\epsilon}( \theta | D, \mathcal{M})$ sera proche de $p( \theta | D, \mathcal{M})$

    L'algorithme ABC naïf s'écrit alors

    ```Python
    thetas = []
    y_hats = []
    for n in range(Nmax) :
        while True
        theta = simuler_theta() # Simuler theta avec la loi sa loi a priori
        y_hat = generer_y(theta) # generer y_hat avec le theta simuler à l'aide du FNN
        if((rho(eta(y_hat), eta(y))) <= epsilon) # On garde de couple si y et y_hat sont assez'similaire' 
            break
        # On collecte les cas favorables pour simuler leur distribution
        thetas.append(theta)
        y_hats.append(theta)
    ```



* #### SS ([Subset Simulation](https://en.wikipedia.org/wiki/Subset_simulation))

    Pour améliorer la convergence de l'algorithme vers les cas favorable (dans la boucle `while`), on procède par échantillonage conditionnés. 

    On peut écrire $\mathbb{P} (\hat{y} \in \mathcal{N}_{\epsilon} (D)| \theta, \mathcal{M}) = \mathbb{P} (\hat{y} \in \mathcal{N}_{\epsilon_1} (D)| \theta, \mathcal{M}) \prod_{i = 2}^m \mathbb{P} (\hat{y} \in \mathcal{N}_{\epsilon_j} (D)| \hat{y} \in \mathcal{N}_{\epsilon_{j-1}} (D), \theta, \mathcal{M})$ avec $ \epsilon = \epsilon_{1} <... < \epsilon_m$

    On a écrit la probabilité difficile à évaluer par un produit de probabilité plus facile à evaluer.

    L'algorithme ABC-SS s'écrit alors

    ```Python
    '''Supposons donnés P0 dans [0,1] choisi tel que N*P0, et 1/P0 sont des entiers et N est la taille des échantillons par itérations'''

    ### On génère l'échantillion initiale
    import numpy as np
    thetas = [simuler_theta() for i in range(0, N)]
    y_hats = [generer_y(X,theta) for theta in thetas]

    for j in range(0, m) : 
        rho_n = []
        for n in range(0, N) :
            rho_n.append(np.linalg.norm(y_hat[n] - y))

        new_indices = list(np.argsort(rho_n))
        thetas = list(np.array(thetas)[new_indices])
        y_hats = list(np.array(y_hats)[new_indices])
        rho_n = list(np.array(rho_n)[new_indices])

        epsilon_j = (rho_n[int(N*P0)] + rho_n[int(N*P0) + 1])/2

        new_thetas = []
        new_y_hats = []
        for k in range(0, int(N*P0)) : 
            t1, t2  = MMA(nb_samples = int(1/P0),seed_theta = thetas[k], tol = epsilon_j, X = X,y = y)

            new_thetas += t1
            new_y_hats += t2

        thetas = new_thetas
        y_hats = new_y_hats
        
        if epsilon_j <= epsilon :
            break     
    ```

* ### Cas étudier
    
    * #### Cas 1

    <img src="photo/fig3_a.png" /> <img src="photo/fig3_b.png" /> 

    * #### Cas 2
    
    <img src="photo/fig5_a.png" /> <img src="photo/fig5_b.png" /> 

<img src = "photo/fig8_a.png"/>


<img src = "photo/fig10_a.png"/>
<img src = "photo/fig10_b.png"/>
<img src = "photo/fig10_c.png"/>